In [1]:
import json
import pandas as pd
from pathlib import Path

DATASET_BASE = Path("/workspace/dataset")
AUDIO_BASE = Path("/mnt/Aimir_HD")

COLLECTIONS = ["suno", "udio"]

FIELDS = {
    "suno": ["upvote_count", "play_count", "is_liked"],
    "udio": ["likes", "plays", "disliked"],
}

collection_dfs = {}

for collection in COLLECTIONS:
    dataset_path = DATASET_BASE / collection
    metadata_path = AUDIO_BASE / collection / "metadata"
    fields = FIELDS[collection]

    song_ids = [p.name for p in sorted(dataset_path.iterdir()) if p.is_dir()]

    records = []
    missing = 0
    for song_id in song_ids:
        json_file = metadata_path / f"{song_id}.json"
        if json_file.exists():
            with open(json_file, "r") as f:
                meta = json.load(f)
            row = {"id": song_id}
            for field in fields:
                row[field] = meta.get(field, None)
            records.append(row)
        else:
            missing += 1

    df = pd.DataFrame(records)
    for field in fields:
        if field != "is_liked":
            df[field] = pd.to_numeric(df[field], errors="coerce").fillna(0).astype(int)
    collection_dfs[collection] = df
    print(f"=== {collection} — {len(df)} songs, missing: {missing} ===")
    print(df[fields].describe())
    print()


=== suno — 19972 songs, missing: 0 ===
       upvote_count     play_count
count  19972.000000   19972.000000
mean       5.252654     204.875275
std      264.004237    6373.309122
min        0.000000       0.000000
25%        0.000000       2.000000
50%        1.000000       4.000000
75%        1.000000      10.000000
max    34921.000000  445490.000000

=== udio — 19992 songs, missing: 0 ===
              likes         plays  disliked
count  19992.000000  19992.000000   19992.0
mean       1.830632     44.095888       0.0
std       12.267748    743.697604       0.0
min        0.000000      0.000000       0.0
25%        0.000000      2.000000       0.0
50%        1.000000      4.000000       0.0
75%        1.000000     16.000000       0.0
max      898.000000  48940.000000       0.0



In [2]:
import pprint

for collection in COLLECTIONS:
    metadata_path = AUDIO_BASE / collection / "metadata"
    sample_file = next(metadata_path.glob("*.json"))
    with open(sample_file, "r") as f:
        sample = json.load(f)
    #print(f"\n=== {collection} — sample metadata ({sample_file.name}) ===")
    #pprint.pprint(sample)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

COLORS = {"Suno": "#FFC400", "Udio": "#FF5B0E"}

metrics = [
    ("Likes", "suno", "upvote_count", "udio", "likes",
     [0, 1, 5, 10, 50, 100]),
    ("Plays", "suno", "play_count",   "udio", "plays",
     [0, 1, 5, 10, 50, 100, 500, 1000]),
]

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams["font.family"] = "Fira Code"

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

stats_rows = []

for ax, (title, sc, sf, uc, uf, tick_values) in zip(axes, metrics):
    frames = []
    for col, field in [("Suno", sf), ("Udio", uf)]:
        s = collection_dfs[col.lower()][field]
        # log1p so zero values don't break the scale
        frames.append(pd.DataFrame({"value": np.log1p(s), "Collection": col}))
        stats_rows.append({"Collection": col, "Metric": title,
                            "Mean": round(s.mean(), 2), "Median": round(s.median(), 2),
                            "Max": int(s.max()), "Std": round(s.std(), 2)})

    df_long = pd.concat(frames, ignore_index=True)

    sns.boxplot(
        data=df_long, x="Collection", y="value",
        palette=COLORS, width=0.45, linewidth=1.0,
        flierprops=dict(marker="o", markersize=2, alpha=0.3,
                        markerfacecolor="#AFAFAF", markeredgecolor="none"),
        ax=ax,
    )

    ax.set_title(title, fontsize=15, fontweight="normal")
    ax.set_xlabel("")
    ax.set_ylabel("count (log scale)")
    ax.tick_params(axis='x', labelsize=15)
    ticks = np.log1p(tick_values)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{v:,}" for v in tick_values])
    ax.set_ylim(0, np.log1p(tick_values[-1]))
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.axhline(0, color="0.8", linewidth=1.0, zorder=0)

plt.tight_layout()
plt.savefig("../Figures/engagement_boxplot.pdf", bbox_inches="tight", dpi=300)
plt.show()

stats_df = pd.DataFrame(stats_rows).set_index(["Metric", "Collection"])
print("\n--- Summary Statistics ---")
print(stats_df.to_string())


In [4]:

# Percentile summary for table
percentiles = [0.90, 0.95]
for collection, fields in [("suno", ("upvote_count", "play_count")), ("udio", ("likes", "plays"))]:
    df = collection_dfs[collection]
    print(f"=== {collection} ===")
    for field in fields:
        vals = df[field].quantile(percentiles)
        print(f"  {field}: 90th={vals[0.90]:.0f}, 95th={vals[0.95]:.0f}")


=== suno ===
  upvote_count: 90th=2, 95th=4
  play_count: 90th=26, 95th=50
=== udio ===
  likes: 90th=3, 95th=8
  plays: 90th=32, 95th=58
